In [1]:
import torch
from torchvision.models import mobilenet_v3_large as mobilenet
from torchvision.models import MobileNet_V3_Large_Weights as pre_weights

from sklearn.metrics import confusion_matrix, accuracy_score
import math
import copy
import numpy as np

from dataset.data_prefetcher import data_prefetcher
from dataset.data_loader import data_loader

### System details

In [2]:
print(f"PyTorch version: {torch.__version__}")

print("--------------------------------------------------")
print(f"Using cuda: {torch.cuda.is_available()}")
print(f"Cuda corrent device: {torch.cuda.current_device()}")
print(f"Cuda device: {torch.cuda.get_device_name(torch.cuda.current_device())}")
print(f"Torch Backend enable: {torch.backends.cudnn.enabled}")
print(f"Torch Backend: {torch.backends.cudnn.version() }")

PyTorch version: 2.2.1+cu121
--------------------------------------------------
Using cuda: True
Cuda corrent device: 0
Cuda device: NVIDIA GeForce GTX 1660 SUPER
Torch Backend enable: True
Torch Backend: 8902


In [3]:
NUM_CLASSES = 2
NUM_EPOCHS = 10

In [4]:
def fast_collate(batch):
    images = [image[0] for image in batch]
    targets = torch.tensor([target[1] for target in batch], dtype=torch.float32)

    width, height = 128, 128
    tensor = torch.zeros((len(images), 3, height, width), dtype=torch.float32).contiguous()

    for index, image in enumerate(images):
        nump_array = np.asarray(image, dtype=np.float32)

        if(nump_array.ndim < 3):
            nump_array = np.expand_dims(nump_array, axis=-1)

        nump_array = np.rollaxis(nump_array, 2)
        tensor[index] += torch.from_numpy(nump_array)

    return tensor, targets

### Getting data

In [5]:
loader = data_loader()

train, val = loader.load_dataset_as_train("/home/tteuz/Desktop/TCC/datasets/PKLot2.0/PKLotSegmented", 0.7)
test = loader.load_dataset_as_test("/home/tteuz/Desktop/TCC/datasets/PKLot2.0/CNRParkEXTSegmented")

In [6]:
collate_fn = lambda batch: fast_collate(batch)

train_loader = torch.utils.data.DataLoader(train, batch_size=64, shuffle=True, num_workers=6, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(val, batch_size=1000, shuffle=False, num_workers=6, collate_fn=collate_fn)

test_loaders = {}
for test_ds in test:
    test_loaders[test_ds] = torch.utils.data.DataLoader(test[test_ds], batch_size=1000, shuffle=False, num_workers=6, collate_fn=collate_fn)

### Fine tunning

In [7]:
model = mobilenet(weights=pre_weights.IMAGENET1K_V2)
model.classifier[-1] = torch.nn.Linear(1280, NUM_CLASSES)

bce_loss = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model.classifier)
print(f"is in cuda: {next(model.parameters()).is_cuda}")
print(f"device: {device}")

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=2, bias=True)
)
is in cuda: True
device: cuda


In [8]:
best_state_dict = None
best_loss = math.inf

In [9]:
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    max_iters = 10
    count = 0

    prefetcher = data_prefetcher(train_loader)
    inputs, labels = prefetcher.next()
    while inputs is not None:
        if count == max_iters:
            break

        inputs, labels = inputs.to(device), labels.type(torch.long).to(device)
        optimizer.zero_grad()

        one_hot_labels = torch.zeros(labels.size(0), 2, device=device)
        one_hot_labels.scatter_(1, labels.unsqueeze(1), 1)
        
        preds = model(inputs)
        loss = bce_loss(torch.sigmoid(preds), one_hot_labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        inputs, labels = prefetcher.next()

        count += 1
    
    epoch_loss = running_loss / len(train)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Training Loss: {epoch_loss:.10f}")

    # Validation 
    model.eval()
    val_labels, val_preds, val_loss = [], [], 0.0
    val_bce_loss = torch.nn.BCELoss()

    max_iters = 5
    count = 0

    prefetcher = data_prefetcher(val_loader)
    inputs, labels = prefetcher.next()
    with torch.no_grad():
        while inputs is not None:
            if count == max_iters:
                break

            inputs, labels = inputs.to(device), labels.type(torch.long).to(device)

            one_hot_labels = torch.zeros(labels.size(0), 2, device=device)
            one_hot_labels.scatter_(1, labels.unsqueeze(1), 1)

            preds = model(inputs)
            loss = val_bce_loss(torch.sigmoid(preds), one_hot_labels)

            val_loss += loss.item() * inputs.size(0)

            val_labels.extend(labels.cpu().numpy())
            _, predicted = torch.max(preds, 1)
            val_preds.extend(predicted.cpu().numpy())

            inputs, labels = prefetcher.next()

            count += 1

    val_loss = val_loss / len(val)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Validation Loss: {val_loss:.10f}")

    if (best_loss - val_loss) > 0.0:
        print("got in here")
        best_state_dict = copy.deepcopy(model.state_dict())
        best_loss = val_loss

    accuracy = accuracy_score(val_labels, val_preds)
    cm = confusion_matrix(val_labels, val_preds)
    
    print(f'Validation Accuracy: {accuracy:.2f}%\n')
    print("----------------------------------------------------------------------")

Epoch [1/10], Training Loss: 0.0001344785
Epoch [1/10], Validation Loss: 0.0191275690
got in here
Validation Accuracy: 0.87%

----------------------------------------------------------------------
Epoch [2/10], Training Loss: 0.0000491511
Epoch [2/10], Validation Loss: 0.0266476297
Validation Accuracy: 0.90%

----------------------------------------------------------------------
Epoch [3/10], Training Loss: 0.0000397553
Epoch [3/10], Validation Loss: 0.0076659016
got in here
Validation Accuracy: 0.92%

----------------------------------------------------------------------
Epoch [4/10], Training Loss: 0.0000429066
Epoch [4/10], Validation Loss: 0.0023509082
got in here
Validation Accuracy: 0.97%

----------------------------------------------------------------------
Epoch [5/10], Training Loss: 0.0000373486
Epoch [5/10], Validation Loss: 0.0031309402
Validation Accuracy: 0.98%

----------------------------------------------------------------------
Epoch [6/10], Training Loss: 0.00002756

### Results

In [ ]:
final_model = mobilenet()
final_model.classifier[-1] = torch.nn.Linear(1280, NUM_CLASSES)
final_model.load_state_dict(best_state_dict)
final_model.to(device)

In [ ]:
final_model.eval()

test_labels = []
test_preds = []

max_iters =5
count = 0

prefetcher = data_prefetcher.data_prefetcher(test_loader)
inputs, labels = prefetcher.next()
with torch.no_grad():
    while inputs is not None:
        if count == max_iters:
            break

        inputs, labels = inputs.to(device), labels.to(device)
        preds = final_model(inputs)

        test_labels.extend(labels.cpu().numpy())
        test_preds.extend((torch.sigmoid(preds).cpu().numpy() > 0.5).astype(int))

        inputs, labels = prefetcher.next()

        count += 1

accuracy = accuracy_score(test_labels, test_preds)
cm = confusion_matrix(test_labels, test_preds)

print(f'Test Accuracy: {accuracy:.2f}%\n')
utils.print_confusion_matrix(cm)